# Arquitectura híbrida de K-means, Random Forest y XGBoost

### para reducir el error del pronóstico de demanda medido en valor en el sector retail

**Maestría en Ciencia de Datos · Universidad Continental · Escuela de Posgrado**

Melo Reyes, Daniel Antonio · Ojeda Collazos, Sharon · Yucasi Quispe, Alexander

---

### Qué es este cuaderno

Presenta los resultados de los tres objetivos específicos de la tesis sobre **cuatro conjuntos de evaluación** provenientes de **tres fuentes independientes**.

El cálculo completo está en `tesis_grupo7.py`, un archivo de 1,835 líneas que ejecuta todo el experimento y guarda sus resultados en disco. Este cuaderno **lee esos resultados y los presenta**: no los vuelve a calcular, porque una corrida completa de los cuatro conjuntos toma unos cuarenta minutos.

Al final hay una sección de reproducción, para quien quiera volver a ejecutarlo y verificar.

**Semilla fijada en 75 en todo el flujo.**

---
## 1. Preparación del entorno

Este cuaderno puede abrirse de dos formas.

**Si ya tiene los archivos en su Drive**, ejecute la celda de montaje. **Si no los tiene**, salte el montaje y use la celda de carga por ZIP que viene después: el cuaderno funciona igual.

Y si solo quiere **leer** los resultados, no necesita ejecutar nada: las tablas y los gráficos quedan guardados dentro del archivo.

In [ ]:
# Solo si tiene los archivos en su propio Drive.
try:
    from google.colab import drive
    drive.mount('/content/drive')
except Exception as e:
    print('Drive no montado. Use la celda de carga por ZIP.', e)

### Localización de los archivos

El cuaderno busca los archivos en varias ubicaciones, de modo que funcione tanto en el Drive del autor como en el de un tercero o desde un ZIP.

In [ ]:
import os, glob
import pandas as pd, numpy as np
import matplotlib.pyplot as plt

pd.set_option('display.max_columns', 40)
pd.set_option('display.width', 200)

CARPETAS = [
    '/content/drive/MyDrive/tesis_grupo7',
    '/content/drive/MyDrive/M5',
    '/content/drive/MyDrive/rohlik',
    '/content/drive/Shareddrives/tesis_grupo7',
    '/content/tesis_grupo7',
    '/content',
    '.',
]

def ruta(nombre):
    """Devuelve la primera ruta existente para 'nombre', o None."""
    for c in CARPETAS:
        p = os.path.join(c, nombre)
        if os.path.exists(p):
            return p
    return None

def hay(nombre):
    return ruta(nombre) is not None

### Carga por ZIP, para quien no tiene los archivos en su Drive

Ejecute esta celda **solo si el montaje no encontró los archivos**. Le pedirá subir `tesis_grupo7_datos.zip`, que contiene los resultados, las figuras y el código.

In [ ]:
# Celda opcional. Ejecutar solo si no se tienen los archivos en el Drive.

# from google.colab import files
# subido = files.upload()
# import zipfile
# for nombre in subido:
#     zipfile.ZipFile(nombre).extractall('/content/tesis_grupo7')
# print('Archivos disponibles en /content/tesis_grupo7')

### Verificación

Antes de continuar se comprueba qué archivos están disponibles.

In [ ]:
ESPERADOS = {
    'Empresa (objetivo 1)'   : 'resultados_objetivo1.csv',
    'Empresa (objetivo 3)'   : 'v6_priv_resultados.csv',
    'M5 (objetivo 1)'        : 'm5_resultados_objetivo1.csv',
    'M5 (objetivo 3)'        : 'v6_m5_oe3.csv',
    'M5 (clústeres)'         : 'v6_m5_clusters.csv',
    'Chequia (objetivo 1)'   : 'roh_cz_objetivo1.csv',
    'Chequia (objetivo 3)'   : 'roh_cz_oe3.csv',
    'Chequia (clústeres)'    : 'roh_cz_clusters.csv',
    'Hungría (objetivo 1)'   : 'roh_hu_objetivo1.csv',
    'Hungría (objetivo 3)'   : 'roh_hu_oe3.csv',
    'Hungría (clústeres)'    : 'roh_hu_clusters.csv',
    'Código del experimento' : 'tesis_grupo7.py',
}
DATOS_CRUDOS = {
    'M5 (panel mensual)'     : 'm5_mensual_item.csv',
    'M5 (precios)'           : 'm5_precios_item.csv',
    'Chequia (panel mensual)': 'rohlik_cz_mensual.csv',
    'Chequia (precios)'      : 'rohlik_cz_precios.csv',
    'Hungría (panel mensual)': 'rohlik_hu_mensual.csv',
    'Hungría (precios)'      : 'rohlik_hu_precios.csv',
}

print('RESULTADOS DEL EXPERIMENTO')
for nombre, f in ESPERADOS.items():
    p = ruta(f)
    print(f'  {nombre:<26} {f:<32} {"disponible" if p else "FALTA"}')

print('\nDATOS DE ORIGEN (públicos; solo para la revisión del objetivo 1)')
for nombre, f in DATOS_CRUDOS.items():
    p = ruta(f)
    print(f'  {nombre:<26} {f:<32} {"disponible" if p else "FALTA"}')

print('\nLas secciones cuyos archivos falten se saltan solas, sin interrumpir el cuaderno.')

---
## 2. Las fuentes y los conjuntos de evaluación

Tres fuentes, cuatro conjuntos. Rohlik genera dos conjuntos porque Chequia y Hungría son mercados independientes: no comparten ningún producto, operan con estructuras de almacén distintas y en monedas distintas. **En ningún cálculo se agregan entre sí.**

In [ ]:
fuentes = pd.DataFrame([
 ['Empresa del sector retail peruano', 'Sistema transaccional propio', 'Privada',
  '2022-01 a 2026-06', '2,109 SKU', 'Mensual', 'PEN'],
 ['Walmart M5', 'Competencia M5', 'Pública',
  '2011-02 a 2016-04', '3,049 ítems', 'Diaria', 'USD'],
 ['Rohlik Chequia', 'Rohlik Group', 'Pública',
  '2020-08 a 2024-05', '958 productos', 'Diaria', 'CZK'],
 ['Rohlik Hungría', 'Rohlik Group', 'Pública',
  '2020-08 a 2024-05', '940 productos', 'Diaria', 'HUF'],
], columns=['Conjunto','Fuente','Acceso','Cobertura','Población','Granularidad original','Moneda'])
fuentes

Los cuatro se homologan a **frecuencia mensual**, con ciclo estacional de doce periodos, para que todos los métodos se evalúen sobre la misma unidad temporal.

---
# OBJETIVO ESPECÍFICO 1

> Realizar el análisis exploratorio de la estructura de la demanda en cada conjunto de datos, identificando patrones de concentración, intermitencia, estacionalidad y variabilidad, así como relaciones y multicolinealidad entre las variables predictoras.

## 3.1 Revisión de los datos de origen

Antes de cualquier modelado se revisa el dato tal como llega.

In [ ]:
def revisar(archivo, nombre):
    p = ruta(archivo)
    if p is None:
        print(f'{nombre}: archivo no disponible en este entorno, se omite.')
        return None
    d = pd.read_csv(p)
    meses = [c for c in d.columns if str(c)[:2] == '20']
    print('='*78)
    print(nombre)
    print('='*78)
    print(f'  Dimensiones        : {d.shape[0]:,} filas x {d.shape[1]} columnas')
    print(f'  Columnas no-mes    : {[c for c in d.columns if c not in meses]}')
    print(f'  Periodos           : {meses[0]} a {meses[-1]}  ({len(meses)} meses)')
    print(f'  Productos únicos   : {d.iloc[:,0].nunique():,}')
    print(f'  Celdas vacías      : {int(d[meses].isna().sum().sum()):,}')
    print(f'  Celdas negativas   : {int((d[meses] < 0).sum().sum()):,}')
    print(f'  Filas duplicadas   : {int(d.iloc[:,0].duplicated().sum()):,}')
    print()
    display(d.head(5).iloc[:, :8])
    return d

M5 = revisar('m5_mensual_item.csv', 'M5 — panel mensual por ítem')

In [ ]:
CZ = revisar('rohlik_cz_mensual.csv', 'ROHLIK CHEQUIA — panel mensual')
HU = revisar('rohlik_hu_mensual.csv', 'ROHLIK HUNGRÍA — panel mensual')

**El conjunto de la empresa no se muestra.** Su archivo está sujeto a condiciones de confidencialidad: no se publican los datos originales ni los identificadores del sistema de origen, y la razón social se mantiene reservada. De ese conjunto se presentan únicamente resultados agregados.

### Precios

El precio se emplea **solo para valorizar las métricas** y para construir la variable `LOG_VALOR`. En ningún conjunto ingresa como predictor.

In [ ]:
for f, n in [('m5_precios_item.csv', 'M5'),
             ('rohlik_cz_precios.csv', 'Chequia'),
             ('rohlik_hu_precios.csv', 'Hungría')]:
    if not hay(f):
        print(f'{n:<10} archivo no disponible, se omite.'); continue
    p = pd.read_csv(ruta(f))
    col = [c for c in p.columns if 'precio' in c.lower()][0]
    print(f'{n:<10} {len(p):>6,} productos | mediana {p[col].median():>10,.2f} | '
          f'mín {p[col].min():>8,.2f} | máx {p[col].max():>12,.2f} | '
          f'no válidos {int((p[col] <= 0).sum())}')

---
## 3.2 Partición temporal

Ninguna variable, ningún criterio de elegibilidad y ninguna decisión del protocolo emplea información posterior al corte de entrenamiento. El conjunto de prueba se usa **únicamente para evaluar**.

In [ ]:
particion = pd.DataFrame([
 ['Empresa', '2022-01 a 2025-06', 42, '2025-07 a 2026-06', 12, '—'],
 ['M5',      '2011-02 a 2015-04', 51, '2015-05 a 2016-04', 12, '2011-01 (3 días) y 2016-05 (22 días)'],
 ['Chequia', '2020-08 a 2023-05', 34, '2023-06 a 2024-05', 12, '2024-06 (2 días, 7.5 % de un mes típico)'],
 ['Hungría', '2020-08 a 2023-05', 34, '2023-06 a 2024-05', 12, '2024-06 (2 días, 7.8 % de un mes típico)'],
], columns=['Conjunto','Entrenamiento','Meses train','Prueba','Meses test','Periodos descartados'])
particion

Los últimos doce meses del entrenamiento se reservan como **ventana de validación interna**, y es sobre ella que se resuelve el torneo entre modelos. El conjunto de prueba no interviene.

---
## 3.3 Concentración del valor

Primer hallazgo del análisis exploratorio: el valor no se reparte de manera uniforme.

In [ ]:
CONJ = {
    'Empresa': 'resultados_objetivo1.csv',
    'M5'     : 'm5_resultados_objetivo1.csv',
    'Chequia': 'roh_cz_objetivo1.csv',
    'Hungría': 'roh_hu_objetivo1.csv',
}
OE1 = {k: pd.read_csv(ruta(v)) for k, v in CONJ.items() if hay(v)}
print('Conjuntos cargados:', ', '.join(OE1))

filas = []
for n, d in OE1.items():
    v = np.sort(d['VALOR'].values)[::-1]
    acum = np.cumsum(v) / v.sum()
    filas.append([n, len(d)] + [f'{(int((acum <= u).sum()) + 1) / len(d):.1%}' for u in (.50, .80, .95)])
pd.DataFrame(filas, columns=['Conjunto','Productos','% de productos que explican el 50 % del valor',
                             '…el 80 %','…el 95 %'])

In [ ]:
fig, ax = plt.subplots(figsize=(7.5, 4.2))
for n, d in OE1.items():
    v = np.sort(d['VALOR'].values)[::-1]
    acum = np.cumsum(v) / v.sum()
    ax.plot(np.arange(1, len(v)+1) / len(v) * 100, acum * 100, linewidth=2, label=n)
ax.axhline(80, color='grey', linestyle='--', linewidth=1)
ax.set_xlabel('Productos ordenados de mayor a menor valor (%)')
ax.set_ylabel('Valor acumulado (%)')
ax.set_title('Concentración del valor en los cuatro conjuntos')
ax.legend(frameon=False); ax.grid(alpha=.3); ax.set_xlim(0, 100); ax.set_ylim(0, 101)
for s in ('top','right'): ax.spines[s].set_visible(False)
plt.show()

Cuanto más pegada al eje izquierdo está la curva, más concentrado es el portafolio. **El conjunto de la empresa es el más concentrado de los cuatro.**

---
## 3.4 Patrones de demanda

Cada producto se clasifica según Syntetos, Boylan y Croston (2005), con dos indicadores calculados **sobre la ventana de entrenamiento**:

- **ADI** — intervalo medio entre demandas: periodos totales dividido entre periodos con venta. Umbral 1.32.
- **CV²** — coeficiente de variación al cuadrado de las cantidades en los periodos con venta. Umbral 0.49.

El cruce de ambos umbrales define cuatro patrones: **suave**, **errático**, **intermitente** y **con picos**. Los dos primeros son los pronosticables por métodos de series regulares y constituyen la muestra.

In [ ]:
filas = []
for n, d in OE1.items():
    t = d['PATRON'].value_counts()
    tot = len(d)
    filas.append([n] + [f'{t.get(p, 0):,} ({t.get(p, 0)/tot:.1%})'
                        for p in ['SUAVE','ERRATICO','INTERMITENTE','CON PICOS','SIN DATOS']])
pd.DataFrame(filas, columns=['Conjunto','Suave','Errático','Intermitente','Con picos','Sin datos'])

In [ ]:
COL = {'SUAVE':'#12876A','ERRATICO':'#2F6F9F','INTERMITENTE':'#C56B2C','CON PICOS':'#7D6A9C'}
fig, axes = plt.subplots(1, 4, figsize=(16, 4), sharey=True)
for ax, (n, d) in zip(axes, OE1.items()):
    e = d.dropna(subset=['ADI','CV2'])
    for p, c in COL.items():
        s = e[e['PATRON'] == p]
        ax.scatter(s['ADI'], s['CV2'], s=10, color=c, alpha=.55, edgecolors='none', label=p.capitalize())
    ax.axvline(1.32, color='grey', linestyle='--', linewidth=1)
    ax.axhline(0.49, color='grey', linestyle='--', linewidth=1)
    ax.set_xscale('log'); ax.set_yscale('log')
    ax.set_title(n); ax.set_xlabel('ADI (log)')
    for s_ in ('top','right'): ax.spines[s_].set_visible(False)
axes[0].set_ylabel('CV² (log)')
axes[0].legend(frameon=False, fontsize=8, loc='lower left')
plt.suptitle('Clasificación de patrones de demanda (Syntetos et al., 2005)', y=1.03)
plt.tight_layout(); plt.show()

Las líneas punteadas son los umbrales. **Abajo a la izquierda** están las series suaves, las más pronosticables. **Arriba a la derecha**, las de picos. La nube de cada conjunto se ubica en un lugar distinto: esa diferencia es el primer resultado de la tesis.

---
## 3.5 Embudo de elegibilidad

De la población total no todos los productos entran a la comparación. El embudo aplica criterios **calculados exclusivamente sobre la ventana de entrenamiento**.

In [ ]:
embudo = pd.DataFrame([
 ['Población',                                    '2,109', '3,049',  '958',  '940'],
 ['Sin venta en la ventana de entrenamiento',       '345',     '0',   '54',  '116'],
 ['Historial menor a 24 meses',                     '432',   '525',  '243',  '336'],
 ['Liquidación o remate',                            '43', 'n/a',  'n/a',  'n/a'],
 ['Precio unitario no válido',                        '0',     '0',    '0',    '0'],
 ['POBLACIÓN DE ESTUDIO',                         '1,289', '2,524',  '661',  '488'],
 ['Descartados por patrón (intermitente o picos)',  '882',   '658',  '137',  '192'],
 ['Descartados por los criterios de negocio',        '83',    '40',   '15',   '18'],
 ['MUESTRA',                                        '324', '1,826',  '509',  '278'],
], columns=['Criterio','Empresa','M5','Chequia','Hungría'])
embudo

Los dos criterios de negocio provienen del sistema de planificación en producción de la empresa y están documentados con anterioridad a esta investigación:

- **Pronosticabilidad** — coeficiente de variación de los periodos con venta del último ciclo menor o igual a 1.00.
- **Actividad reciente** — venta en al menos 6 periodos del último ciclo y en al menos 4 de los últimos 6.

Un producto elegible al cierre del entrenamiento **permanece en la evaluación aunque registre venta cero durante los doce periodos de prueba**. Consultar el conjunto de prueba para decidir quién entra a la muestra introduciría información del futuro en la definición del objeto evaluado.

---
## 3.6 El hallazgo que sustenta el problema

La segmentación por valor de uso que emplea la práctica empresarial no considera la variabilidad de la demanda. La consecuencia se mide.

In [ ]:
filas = []
for n, d in OE1.items():
    corte = d['VALOR'].quantile(0.80)
    top = d[d['VALOR'] >= corte]
    duros = top[top['PATRON'].isin(['INTERMITENTE','CON PICOS'])]
    filas.append([n, len(top), len(duros), f'{duros["VALOR"].sum()/d["VALOR"].sum():.1%}'])
pd.DataFrame(filas, columns=['Conjunto','Productos del 20 % superior de valor',
                             'De ellos, NO pronosticables por métodos regulares',
                             'Valor que representan'])

Son productos que el esquema actual prioriza por su valor, pero cuya demanda **no es pronosticable** con los métodos que ese mismo esquema aplica. Ese desencuentro es el problema que la tesis plantea.

---
## 3.7 Relación entre las variables predictoras

Las once variables se construyen a partir de la propia serie, con información anterior al corte de entrenamiento: rezagos, medias móviles, frecuencia, variabilidad, estacionalidad, crecimiento interanual, valor y autocorrelación.

Construir rezagos y medias sobre una misma serie genera **por diseño** predictores redundantes. El diagnóstico se reporta como caracterización del conjunto de variables, no como supuesto que condicione la validez del modelado: los ensambles de árboles no dependen de la estimación de coeficientes.

In [ ]:
vif = pd.DataFrame([
 ['MEDIA_12',  'Promedio de los últimos doce meses',              43.64, 48.79],
 ['LAG_1',     'Unidades del mes anterior',                       14.99, 19.72],
 ['MEDIA_3',   'Promedio de los últimos tres meses',              14.13, 41.02],
 ['LAG_12',    'Unidades del mismo mes del año anterior',          8.51, 12.69],
 ['CV_12',     'Coeficiente de variación de los últimos 12 meses', 1.97,  1.87],
 ['ESTAC',     'Intensidad del patrón estacional',                 2.04,  1.29],
 ['FREC_12',   'Meses con venta en el último ciclo',               1.77,  1.57],
 ['LOG_VALOR', 'Logaritmo del valor comercializado',               1.56,  1.49],
 ['YOY',       'Variación interanual',                             1.24,  1.08],
 ['AUTOCORR',  'Autocorrelación de orden uno',                     1.07,  1.07],
], columns=['Variable','Descripción','FIV Empresa','FIV M5'])
vif

Las cuatro primeras superan el umbral de 10 que la literatura considera severo, y era esperable: `MEDIA_12`, `MEDIA_3`, `LAG_1` y `LAG_12` describen el mismo nivel de la serie en ventanas distintas.

---
# OBJETIVO ESPECÍFICO 2

> Construir la arquitectura híbrida de pronóstico, segmentando los SKU elegibles de cada conjunto mediante K-means y asignando a cada clúster, por validación temporal, el modelo de mejor desempeño entre Random Forest y XGBoost.

## 4.1 Selección del número de grupos

El número de grupos **no se elige a criterio del investigador**. Se decide con tres reglas declaradas antes de mirar el conjunto de prueba:

1. **Mayor índice de Rand ajustado** sobre veinte remuestreos del 80 % de la muestra. Mide si la partición se sostiene cuando cambian los datos.
2. **Desempate por índice de silueta** cuando dos candidatos quedan dentro de 0.01.
3. **Piso de negocio de K ≥ 3.** El sistema en producción mantiene cuatro clústeres, de los cuales tres reciben modelo de pronóstico y el cuarto se gobierna por cuota comercial. Una partición con menos de tres grupos no permitiría la asignación diferenciada que exige el objetivo.

In [ ]:
seleccion = pd.DataFrame([
 ['Empresa', 3, 0.910, 0.217, 'K-means frente a jerárquico y mezclas gaussianas'],
 ['M5',      4, 0.940, 0.236, 'Empate dentro de 0.01 en K = 3, 4, 6 y 7; desempate por silueta'],
 ['Chequia', 5, 0.880, 0.219, 'Empate en K = 5 y 7; el piso de negocio excluye K = 2'],
 ['Hungría', 6, 0.865, 0.229, 'Máximo sin empate dentro del margen'],
], columns=['Conjunto','K elegido','Rand ajustado','Silueta','Observación'])
seleccion

**Un dato que la tesis reporta como limitación, no como logro:** la silueta se mueve entre 0.217 y 0.236. Son particiones **reproducibles pero moderadamente separadas**. El Rand ajustado alto dice que los grupos se sostienen ante remuestreo; la silueta baja dice que sus fronteras no son nítidas. Ambas cosas se reportan.

## 4.2 Perfil de los grupos obtenidos

Los grupos se construyen sobre cinco variables de comportamiento estandarizadas: variabilidad del ciclo, intensidad estacional, variación interanual, logaritmo del valor y autocorrelación.

In [ ]:
perfil = pd.DataFrame([
 ['Empresa', 3, 'Núcleo de alto valor', 106, '77.2 %'],
 ['M5',      4, 'Núcleo de alto valor', 882, '60.6 %'],
 ['Chequia', 5, 'Núcleo de alto valor', 116, '41.0 %'],
 ['Hungría', 6, 'Núcleo de alto valor',  49, '35.8 %'],
], columns=['Conjunto','Grupos','Clúster de mayor peso','SKU','% del valor (entrenamiento)'])
perfil

El peso económico de cada clúster se calcula **exclusivamente sobre la ventana de entrenamiento**. Calcularlo sobre la serie completa introduciría información del periodo de prueba en la definición del objeto evaluado, que es justamente el clúster sobre el que se contrasta la hipótesis específica 3.

**El número de grupos cambia en cada portafolio: 3, 4, 5 y 6.** Ese es un resultado, no un accidente: la estructura no es transferible entre portafolios.

## 4.3 Torneo entre Random Forest y XGBoost

En cada grupo compiten los dos algoritmos sobre una rejilla de hiperparámetros, y gana el de menor **WMAPE valorizado en la ventana de validación**. El conjunto de prueba no interviene.

In [ ]:
torneo = pd.DataFrame([
 ['Empresa', 3, 3, 0, '—'],
 ['M5',      4, 3, 1, 'XGBoost gana el núcleo por 0.1 pp (15.1 % contra 15.2 %)'],
 ['Chequia', 5, 5, 0, 'Empate en 29.7 % en el grupo 5, resuelto por la regla declarada'],
 ['Hungría', 6, 5, 1, 'XGBoost gana el grupo 2 por 0.7 pp (18.0 % contra 18.7 %)'],
 ['TOTAL',  18, 16, 2, ''],
], columns=['Conjunto','Clústeres','Gana Random Forest','Gana XGBoost','Observación'])
torneo

**Dieciocho clústeres. Random Forest gana dieciséis, XGBoost dos, y en ambos casos por márgenes mínimos.**

La lectura correcta de este resultado: el aporte del torneo no consiste en repartir algoritmos distintos entre grupos, sino en **determinar por evidencia, y no por supuesto, cuál corresponde a cada uno**. La diferenciación se expresa principalmente en los hiperparámetros, que sí son propios de cada grupo.

---
# OBJETIVO ESPECÍFICO 3

> Evaluar en qué medida la arquitectura híbrida reduce el WMAPE valorizado frente al promedio móvil y a Holt-Winters en el clúster de mayor peso económico de cada conjunto de datos evaluado.

## 5.1 Las dos líneas base

**Promedio móvil.** Descomposición clásica multiplicativa: nivel del último ciclo, índice estacional y factor de crecimiento calculado sobre **años civiles cerrados**. Es la lógica del planificador en producción y representa la práctica empresarial descrita en el planteamiento del problema.

**Holt-Winters.** Suavizamiento exponencial con tendencia y estacionalidad aditivas. Es la referencia estadística clásica frente a la cual la literatura evalúa las propuestas de pronóstico.

Se incluyen dos y no una para que una eventual ventaja de la arquitectura no pueda atribuirse a la debilidad de una única línea base.

## 5.2 Las métricas

In [ ]:
metricas = pd.DataFrame([
 ['WMAPE valorizado', 'Contraste',
  'Suma del error absoluto de cada SKU ponderado por su precio, sobre la venta real valorizada. Es el indicador con el que se contrasta la hipótesis.'],
 ['BIAS valorizado', 'Diagnóstico complementario',
  'Pronóstico valorizado sobre venta real valorizada, menos uno. Positivo indica sobrepronóstico. No forma parte de ninguna hipótesis.'],
 ['WMAPE en unidades', 'Complementaria', 'La misma magnitud sin valorizar.'],
 ['R²', 'Complementaria', 'Proporción de la variabilidad explicada.'],
 ['Wilcoxon', 'Contraste estadístico',
  'Prueba de rangos con signo, bilateral, sobre el error absoluto valorizado de cada SKU, independiente por clúster.'],
], columns=['Métrica','Rol','Definición'])
metricas

## 5.3 Desempeño en el conjunto de prueba

Los doce meses de prueba no intervinieron en ninguna etapa del entrenamiento ni en la resolución del torneo.

In [ ]:
def tabla_oe3(f, nombre):
    if not hay(f):
        print(f'{nombre}: resultados no disponibles, se omite.'); return
    d = pd.read_csv(ruta(f))
    d = d[d['grupo'] != 'AGREGADO'].copy()
    d['% valor']  = (d['peso'] * 100).round(1).astype(str) + ' %'
    d['WMAPE $']  = (d['wmape_val'] * 100).round(1).astype(str) + ' %'
    d['BIAS $']   = (d['bias_val'] * 100).round(1).map(lambda x: f'{x:+.1f} %')
    d['R²']       = d['r2'].round(2)
    salida = d[['grupo','% valor','metodo','WMAPE $','BIAS $','R²','p_pm','p_hw']]
    salida.columns = ['Grupo','% del valor','Método','WMAPE $','BIAS $','R²','p vs PM','p vs HW']
    print(f'\n{nombre}')
    display(salida.set_index(['Grupo','% del valor','Método']))

tabla_oe3('v6_m5_oe3.csv',  'WALMART M5')

In [ ]:
tabla_oe3('roh_cz_oe3.csv', 'ROHLIK CHEQUIA')
tabla_oe3('roh_hu_oe3.csv', 'ROHLIK HUNGRÍA')

In [ ]:
p = pd.read_csv(ruta('v6_priv_resultados.csv'))
p['Método']      = p['metodo'].map({'PM':'Promedio móvil','HW':'Holt-Winters','ML':'Arquitectura'})
p['% del valor'] = (p['peso'] * 100).round(1).astype(str) + ' %'
p['WMAPE $']     = (p['wv'] * 100).round(1).astype(str) + ' %'
p['BIAS $']      = (p['bv'] * 100).round(1).map(lambda x: f'{x:+.1f} %')
p['R²']          = p['r2'].round(2)
print('\nEMPRESA DEL SECTOR RETAIL PERUANO  (resultados agregados; el dato de origen es confidencial)')
display(p[['nombre','% del valor','Método','WMAPE $','BIAS $','R²']]
        .set_index(['nombre','% del valor','Método']))

## 5.4 Comparación gráfica por clúster

In [ ]:
ORD = ['Promedio móvil','Holt-Winters','Arquitectura']
CC  = {'Promedio móvil':'#2F6F9F','Holt-Winters':'#C56B2C','Arquitectura':'#12876A'}

fig, axes = plt.subplots(1, 3, figsize=(16, 4.4))
for ax, (f, n) in zip(axes, [('v6_m5_oe3.csv','M5'), ('roh_cz_oe3.csv','Chequia'), ('roh_hu_oe3.csv','Hungría')]):
    d = pd.read_csv(ruta(f)); d = d[d['grupo'] != 'AGREGADO']
    grupos = d['grupo'].unique()
    x = np.arange(len(grupos)); w = 0.26
    for i, m in enumerate(ORD):
        v = [d[(d['grupo'] == g) & (d['metodo'] == m)]['wmape_val'].values[0] * 100 for g in grupos]
        ax.bar(x + (i-1)*w, v, w, color=CC[m], label=m)
    ax.set_xticks(x); ax.set_xticklabels([g.split('.')[0] for g in grupos])
    ax.set_xlabel('Clúster'); ax.set_title(n); ax.grid(axis='y', alpha=.3); ax.set_axisbelow(True)
    for s in ('top','right'): ax.spines[s].set_visible(False)
axes[0].set_ylabel('WMAPE valorizado (%)')
axes[0].legend(frameon=False, fontsize=9)
plt.suptitle('WMAPE valorizado por clúster y método, conjunto de prueba', y=1.02)
plt.tight_layout(); plt.show()

**Barra más baja es mejor.** El clúster 1 de cada conjunto es el de mayor peso económico y es el objeto de la hipótesis específica 3.

---
## 6. Síntesis de los cuatro conjuntos

Resultado en el clúster de mayor peso económico de cada conjunto. Las cifras están en la moneda de cada conjunto y **no son comparables entre sí en términos absolutos**: lo comparable es la posición relativa de los tres métodos dentro de cada conjunto.

In [ ]:
sintesis = pd.DataFrame([
 ['Empresa', '324', 3, '106', '77.2 %', '32.8 %', '33.4 %', '33.8 %', '0.012', '0.005'],
 ['M5',    '1,826', 4, '882', '60.6 %', '23.8 %', '31.4 %', '26.7 %', '< 0.001', '< 0.001'],
 ['Chequia', '509', 5, '116', '41.0 %', '32.4 %', '34.9 %', '34.9 %', '0.163', '0.063'],
 ['Hungría', '278', 6,  '49', '35.8 %', '42.6 %', '44.9 %', '51.0 %', '0.285', '0.030'],
], columns=['Conjunto','Muestra','K','SKU del clúster','% del valor',
            'Arquitectura','Promedio móvil','Holt-Winters','p vs PM','p vs HW'])
sintesis

### Cómo se lee esta tabla

**La arquitectura obtiene el menor WMAPE valorizado en los cuatro conjuntos.** Eso se verifica en magnitud, sin excepción.

**La significancia estadística no acompaña de manera uniforme:**

| | frente al promedio móvil | frente a Holt-Winters |
|---|---|---|
| Empresa | significativo | significativo |
| M5 | significativo | significativo |
| Chequia | no significativo | no significativo |
| Hungría | no significativo | significativo |

La hipótesis específica 3 **se declara cumplida en su formulación y parcialmente respaldada en su contraste estadístico**, y así se reporta.

## 6.1 El resultado adverso: el sesgo

El BIAS valorizado **no mejora de manera consistente**, y el estudio lo reporta como tal.

In [ ]:
bias = pd.DataFrame([
 ['Empresa',  '−7.3 %', '−8.2 %', '−9.8 %', 'La arquitectura obtiene el sesgo más cercano a cero'],
 ['M5',       '−6.4 %', '+2.6 %', '−3.6 %', 'No lo obtiene'],
 ['Chequia',  '+5.4 %', '−0.9 %', '−3.0 %', 'No lo obtiene; sobrepronostica'],
 ['Hungría', '+17.5 %', '+1.6 %', '+11.1 %', 'No lo obtiene; sobrepronostica'],
], columns=['Conjunto','Arquitectura','Promedio móvil','Holt-Winters','Lectura'])
bias

Una verificación adicional confirmó que ese sesgo **no puede corregirse** mediante un factor estimado en la ventana de validación, porque su signo se invierte entre validación y prueba en dos de los tres grupos del conjunto de la empresa: el factor corregiría en la dirección equivocada.

Esa es la razón por la que el BIAS valorizado **se retiró de la hipótesis** y se conserva como métrica complementaria de diagnóstico. Los datos no sostienen que la reducción del sesgo sea una propiedad de la arquitectura, y declararla como tal sería afirmar más de lo que la evidencia permite.

## 6.2 Magnitud económica

El estudio **no mide** el efecto sobre los quiebres de stock ni sobre el sobreinventario. Lo que sí cuantifica es el error valorizado que la arquitectura deja de cometer.

In [ ]:
economia = pd.DataFrame([
 ['Clúster de mayor peso (106 SKU, 77.2 % del valor)', '+67,243', '+111,155'],
 ['Muestra completa (324 SKU)',                       '+293,562', '+262,566'],
], columns=['Alcance','Frente al promedio móvil','Frente a Holt-Winters'])
economia

Unidades monetarias del conjunto de la empresa, durante los doce meses de evaluación.

Estas cifras expresan una **menor exposición económica al error de pronóstico**. No expresan un efecto sobre el nivel de inventario ni sobre la continuidad del abastecimiento, variables que este diseño no mide y sobre las cuales el estudio no formula afirmación alguna.

---
## 7. Las seis figuras del conjunto de la empresa

Corresponden al Capítulo V del documento. Se generan al ejecutar el conjunto privado y quedan guardadas en la carpeta `figuras_v6`.

In [ ]:
from IPython.display import Image, display

TITULOS = {
 'fig1_concentracion.png': 'Figura 1 — Concentración del valor en el portafolio',
 'fig2_patrones.png'     : 'Figura 2 — Clasificación de patrones de demanda',
 'fig3_correlacion.png'  : 'Figura 3 — Matriz de correlaciones entre las variables predictoras',
 'fig4_bivariados.png'   : 'Figura 4 — Relación bivariada de las principales predictoras',
 'fig5_grupos.png'       : 'Figura 5 — Perfil de los grupos obtenidos',
 'fig6_wmape.png'        : 'Figura 6 — WMAPE valorizado por grupo y método',
}
base = None
for c in CARPETAS:
    if os.path.isdir(os.path.join(c, 'figuras_v6')):
        base = os.path.join(c, 'figuras_v6'); break
encontradas = sorted(glob.glob(os.path.join(base, '*.png'))) if base else []
if not encontradas:
    print('No hay figuras en figuras_v6. Se generan al ejecutar el conjunto privado (ver sección 9).')
for f in encontradas:
    print('\n' + TITULOS.get(os.path.basename(f), os.path.basename(f)))
    display(Image(f, width=720))

**Sobre las Figuras 3 y 4.** Contrastan las variables predictoras con la venta del periodo de prueba. Ese contraste es **descriptivo y posterior al modelado**: la variable de venta no es un predictor y la relación mostrada no intervino en la selección de variables ni en ninguna decisión del protocolo, que se resolvieron con la ventana de entrenamiento.

---
## 8. Trazabilidad de los datos públicos

Los archivos que consume este experimento no son los originales de las competencias: son paneles mensuales derivados. La transformación está documentada en dos cuadernos.

In [ ]:
traza = pd.DataFrame([
 ['M5', 'm5.ipynb',
  'calendar.csv + sales_train_evaluation.csv + sell_prices.csv',
  'm5_mensual_item.csv, m5_precios_item.csv',
  'Suma las 10 tiendas por item_id y agrega a mes calendario. Salida: 3,049 ítems x 65 meses.'],
 ['Rohlik', 'Rohlik.ipynb',
  'rohlik-sales-forecasting-challenge-v2.zip',
  'rohlik_{cz,hu}_mensual.csv, rohlik_{cz,hu}_precios.csv',
  'Asigna Praga y Brno a Chequia y Budapest a Hungría, suma la venta diaria a mes por producto y calcula el precio medio ponderado.'],
], columns=['Fuente','Cuaderno','Archivos originales','Archivos derivados','Qué hace'])
traza

No es necesario volver a procesar los archivos originales para reproducir los resultados: los paneles derivados están en el Drive.

**Una precisión declarada:** el precio medio de Rohlik se calcula sobre todo el historial, periodo de prueba incluido. El precio solo valoriza las métricas y alimenta la variable `LOG_VALOR`; no entra como predictor.

---
## 9. Reproducción del experimento

Todo lo anterior son resultados **ya calculados**. Para volver a generarlos desde cero, esta sección ejecuta el experimento completo.

**Antes de ejecutar, fijar las versiones.** Una diferencia de versión cambia los resultados.

In [ ]:
# Celda opcional. Correr solo si se va a reproducir el experimento.
# Al terminar, Colab pedirá reiniciar el entorno: hacerlo y volver a montar el Drive.

# !pip -q install "numpy==2.4.4" "pandas==3.0.2" "scipy==1.17.1" \
#                 "scikit-learn==1.8.0" "xgboost==3.2.0" "statsmodels==0.15.0"

import sys, numpy, scipy, sklearn, xgboost, statsmodels
entorno = pd.DataFrame([
 ['Python', '3.13', '.'.join(sys.version.split()[0].split('.')[:2])],
 ['NumPy', '2.4.4', numpy.__version__],
 ['pandas', '3.0.2', pd.__version__],
 ['SciPy', '1.17.1', scipy.__version__],
 ['scikit-learn', '1.8.0', sklearn.__version__],
 ['XGBoost', '3.2.0', xgboost.__version__],
 ['statsmodels', '0.15.0', statsmodels.__version__],
], columns=['Componente','Versión de referencia','Versión actual'])
entorno['Estado'] = np.where(entorno['Versión de referencia'] == entorno['Versión actual'], 'coincide', 'DISTINTA')
entorno

### Ejecutar

Un conjunto por celda. Tiempos aproximados: Hungría 3 minutos, Chequia 4, la empresa 10, M5 18.

In [ ]:
# !python tesis_grupo7.py rohlik_hu

In [ ]:
# !python tesis_grupo7.py rohlik_cz

In [ ]:
# !python tesis_grupo7.py m5

In [ ]:
# !python tesis_grupo7.py privado

### Qué comprobar al reproducir

| Verificación | Valor esperado |
|---|---|
| Tamaño de muestra | 324, 1,826, 509 y 278 |
| Número de grupos | 3, 4, 5 y 6 |
| Ganador del torneo | 16 Random Forest y 2 XGBoost en 18 clústeres |
| WMAPE del clúster principal | según la tabla de síntesis |
| Reducción frente al promedio móvil | +293,562 en la muestra de la empresa |

**Una advertencia sobre Holt-Winters.** Ese método estima sus parámetros por optimización numérica y **no admite fijación por semilla**. En los conjuntos cuya ventana de entrenamiento es más corta puede variar unas décimas entre entornos, y en algunas series el optimizador ni siquiera converge. La arquitectura y el promedio móvil, que sí dependen de la semilla, reproducen de forma exacta.

Este comportamiento está documentado en el Capítulo V como **error aleatorio declarado**, no como una diferencia de procedimiento.

---

## 10. Ver el código completo del experimento

In [ ]:
from IPython.display import Code
p = ruta('tesis_grupo7.py')
Code(filename=p, language='python') if p else print('tesis_grupo7.py no disponible en este entorno.')

---

### Alcance y limitaciones declaradas

**Confidencialidad.** El acceso a los datos de la empresa del sector retail peruano está sujeto a condiciones de confidencialidad: no se publican los datos originales ni los identificadores del sistema de origen, y la razón social se mantiene reservada. Las otras dos fuentes son de acceso público, lo que permite la verificación independiente de esa parte del estudio.

**Cuatro condiciones acotan el alcance de los resultados:**

1. **Potencia estadística.** Las muestras de la empresa, Chequia y Hungría comprenden 324, 509 y 278 productos. Los clústeres de menor tamaño no alcanzan significancia pese a presentar diferencias favorables en magnitud.
2. **Separación moderada de los grupos.** La partición es reproducible ante remuestreo, pero sus fronteras no son nítidas.
3. **Ventana corta en Rohlik.** Treinta y cuatro periodos, con solo dos años civiles cerrados, lo que limita a un único ratio el factor de crecimiento del promedio móvil.
4. **Exclusión de precio, promoción y disponibilidad.** M5 y Rohlik las publican, el conjunto de la empresa no las registra. Preservar la comparabilidad del protocolo tuvo el costo de evaluar la arquitectura en un escenario informativamente conservador.